# Init

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

cust_az12_df = spark.read.format("delta").table("bike_lakehouse.bronze.cust_az12_bronze")
cust_az12_df.display()


## Transformation

- Change column name 
- Clean NAS from CID


In [0]:
COLUMN_NAME = {
    "CID": "customer_key",
    "BDATE": "date_of_birth",
    "GEN": "gender"
}

### Change the column names 

In [0]:
for old_name, new_name in COLUMN_NAME.items():
    cust_az12_df = cust_az12_df.withColumnRenamed(old_name, new_name)
cust_az12_df.display()

### Normalize the customer_key Values

In [0]:
cust_az12_df = cust_az12_df.withColumn("customer_key", regexp_replace(col("customer_key"), "NAS", ""))

In [0]:
customer_dob_information = cust_az12_df.withColumn("gender", trim(col("gender")))

In [0]:
customer_dob_information_silver = customer_dob_information.withColumn(
    "gender",
    when(col("gender").isin("M", "Male"), "Male")
    .when(col("gender").isin("F", "Female"), "Female")
    .otherwise("n/a")
)

## Write table to silver layer

In [0]:
customer_dob_information_silver.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable("bike_lakehouse.silver.customer_dob_information_silver")